# prepare_data


In [ ]:
from pathlib import Path
import sys
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
if str(_root / '.runtime') not in sys.path: sys.path.insert(0, str(_root / '.runtime'))
__file__ = str(_root / 'prepare_data.py')
__package__ = ''
__name__ = 'prepare_data'


In [ ]:
"""Verify and extract the complete author-provided dataset, then inventory splits."""
import json
import shutil
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent
EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}




In [ ]:
def prepare():
    archive = ROOT / "downloads/UnderWaterDataset.zip"
    if not archive.is_file():
        raise FileNotFoundError(f"Dataset archive is not ready: {archive}")
    with zipfile.ZipFile(archive) as data:
        required = sum(item.file_size for item in data.infolist())
        if shutil.disk_usage(ROOT).free < required + 1024**3:
            raise RuntimeError("Not enough free space to extract the dataset safely")
        for item in data.infolist():
            if not (ROOT / item.filename).resolve().is_relative_to(ROOT):
                raise ValueError(f"Unsafe ZIP member: {item.filename}")
        print(f"Archive: {len(data.infolist())} entries, {required / 1024**3:.2f} GiB uncompressed", flush=True)
        bad = data.testzip()
        if bad:
            raise ValueError(f"Corrupt ZIP member: {bad}")
        # The published ZIP contains dataset folders directly, without the README's wrapper.
        wrapped = any(name.startswith("UnderWaterDataset/") for name in data.namelist())
        destination = ROOT if wrapped else ROOT / "UnderWaterDataset"
        destination.mkdir(exist_ok=True)
        data.extractall(destination)
    return inventory_dataset()




In [ ]:
def inventory_dataset():
    dataset_root = ROOT / "UnderWaterDataset"
    if not dataset_root.is_dir():
        raise RuntimeError("Archive layout differs from README. Inspect extracted folders before training.")
    inventory = {}
    for folder in sorted(dataset_root.iterdir()):
        if not folder.is_dir():
            continue
        splits = {}
        for split in ("train", "val", "test"):
            inputs = folder / split / "input"
            targets = folder / split / "GT"
            if inputs.is_dir() and targets.is_dir():
                source_names = {p.name for p in inputs.iterdir() if p.suffix.lower() in EXTENSIONS}
                target_names = {p.name for p in targets.iterdir() if p.suffix.lower() in EXTENSIONS}
                if source_names != target_names:
                    raise ValueError(f"Unmatched image/GT filenames: {folder.name}/{split}")
                if not source_names:
                    raise ValueError(f"Empty split: {folder.name}/{split}")
                splits[split] = {"paired_images": len(source_names)}
        if not splits:
            splits["all"] = {"images": sum(p.suffix.lower() in EXTENSIONS for p in folder.rglob("*"))}
        inventory[folder.name] = splits
    (ROOT / "downloads/dataset_inventory.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")
    print(json.dumps(inventory, indent=2), flush=True)
    return inventory


